# Data Cleaning

We utilize only the first version of the Geneva-Copenhagen Survey (GCS I), focusing on F- and G-type stars.

### Key Reference: Holmberg et al., 2007

* **Paper:** [The Geneva-Copenhagen survey of the Solar neighborhood: Ages, metallicities, and kinematic properties of ~14,000 F and G dwarfs](https://www.aanda.org/articles/aa/abs/2004/18/aa0959/aa0959.html)
* **Data Access (Vizier):** [Geneva-Copenhagen Survey of the Solar neighborhood: V/117A](https://cdsarc.cds.unistra.fr/viz-bin/cat/V/117A#/browse)

To view the parameter descriptions, you can refer to the [ReadMe](https://cdsarc.cds.unistra.fr/ftp/V/117A/ReadMe) file from the Geneva-Copenhagen Survey I (Holmberg et al., 2007).

This notebook applies the selection criteria described in Section 2 of the paper, cross-matches with SIMBAD, and exports three CSV files.

## 1. Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

import warnings
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.coordinates import Angle

warnings.simplefilter("ignore", category=fits.verify.VerifyWarning)

In [2]:
ROOT = Path("../")
FITS_PATH = ROOT / "data" / "raw" / "gcs1.fits"
SIMBAD_FILE = ROOT / "data" / "raw" / "resources" / "simbad_query.txt"
STAR_NAMES = ROOT / "data" / "raw" / "resources" / "stars_names.txt"
OUT_ALL = ROOT / "data" / "processed" / "gcs-allstars.csv"
OUT_F = ROOT / "data" / "processed" / "gcs-Fstars.csv"
OUT_G = ROOT / "data" / "processed" / "gcs-Gstars.csv"

# Columns to keep in the final dataset
SELECTED_COLUMNS = [
    "Name",
    "fs",
    "RAh",
    "RAm",
    "RAs",
    "DE_",
    "DEd",
    "DEm",
    "DEs",
    "GLON",
    "GLAT",
    "Vmag",
    "b_y",
    "Hbeta",
    "E_b_y_",
    "logTe",
    "_Fe_H_",
    "Dist",
    "VMAG",
    "dVMag",
    "Age",
    "clAge",
    "chAge",
    "mass",
    "clmass",
    "chmass",
    "RVel",
    "meRVel",
    "e_RVel",
    "o_RVel",
    "dT",
    "P_chi2_",
    "vsini",
    "pmRA",
    "pmDE",
    "e_pm",
    "plx",
    "e_plx",
    "UVel",
    "VVel",
    "WVel",
    "Rgal",
    "zgal",
    "Rmin",
    "Rmax",
    "ecc",
    "zmax",
    "RA",
    "DEC",
    "X",
    "Y",
    "Z",
    "obj_type",
    "spec_type",
]

COLUMN_RENAME = {
    "E_b_y_": "E(b-y)",
    "_Fe_H_": "[Fe/H]",
    "P_chi2_": "P(chi2)",
    "obj_type": "ObjType",
    "spec_type": "SpecType",
}

## 2. Load FITS catalogue

In [3]:
with fits.open(FITS_PATH) as hdul:
    df = pd.DataFrame.from_records(hdul[1].data)

print(f"Raw catalogue: {len(df):,} stars with {df.shape[1]} columns")
print(f"Stars with vsini == 0: {(df['vsini'] == 0).sum():,}")
print(
    f"Mass range (non-zero): {df[df['mass'] > 0]['mass'].min():.2f} - {df[df['mass'] > 0]['mass'].max():.2f} Msun"
)

Raw catalogue: 16,682 stars with 56 columns
Stars with vsini == 0: 3,884
Mass range (non-zero): 0.58 - 2.43 Msun


## 3. Coordinate conversion

Equatorial coordinates to decimal degrees. Galactic XYZ positions (equations 1 - 3 of the paper):
$$
X = R\cos(b)\cos(\ell), \quad Y = R\cos(b)\sin(\ell), \quad Z = R\sin(b)
$$

In [4]:
def _ra_deg(row):
    return Angle(f"{int(row.RAh)} {int(row.RAm)} {row.RAs}", unit="hourangle").degree


def _dec_deg(row):
    sign = row["DE_"].strip()
    return Angle(f"{sign}{int(row.DEd)} {int(row.DEm)} {row.DEs}", unit="degree").degree


df["RA"] = df.apply(_ra_deg, axis=1)
df["DEC"] = df.apply(_dec_deg, axis=1)

# Galactic Cartesian coordinates [pc]
lat_r = np.radians(df["GLAT"])
lon_r = np.radians(df["GLON"])
df["X"] = df["Dist"] * np.cos(lat_r) * np.cos(lon_r)
df["Y"] = df["Dist"] * np.cos(lat_r) * np.sin(lon_r)
df["Z"] = df["Dist"] * np.sin(lat_r)

## 4. Quality cuts

| Cut | Flag | 
|-----|------|
| Binaries | `fb`, `fd`, `Comp` | 
| Suspected giants | `fg` | 
| Undefined vsini | `vsini == 0` |
| Undefined distance | `Dist == 0` |
| Undefined Teff | `logTe == 0` |

In [5]:
n0 = len(df)

# Remove binaries (fb=*, fd=*, or part of a multiple system)
mask_bin = (df["fb"] != "*") & (df["fd"] != "*") & (df["Comp"].str.strip() == "")
df = df[mask_bin].copy()
print(f"After binary removal: {len(df):,} ({n0 - len(df):,} removed)")

# Remove suspected giants
mask_giant = df["fg"] != "*"
df = df[mask_giant].copy()
print(f"After giant removal: {len(df):,} ({n0 - len(df):,} removed)")

# Keep only stars with valid measurements
df = df[(df["vsini"] != 0) & (df["Dist"] != 0) & (df["logTe"] != 0)].copy()
print(f"After measurement cuts: {len(df):,} ({n0 - len(df):,} removed)")

After binary removal: 9,935 (6,747 removed)
After giant removal: 9,709 (6,973 removed)
After measurement cuts: 6,768 (9,914 removed)


## 5. SIMBAD cross-match

For the spectral class for each of the sample spectra, we query the identifier in SIMBAD database and check the availability of the stellar classification and object type. According to the [list of object types](https://simbad.cds.unistra.fr/guide/otypes.htx), which is also saved in `data/otypes.list`, we can see that our sample includes the following:

| Object Type | Description                  |
|-------------|------------------------------|
| PM*         | High Proper Motion Star      |
| *           | Star                         |
| SB*         | Spectroscopic Binary         |
| Em*         | Emission-line Star           |
| **          | Double or Multiple Star      |
| BY*         | BY Dra Variable              |
| Er*         | Eruptive Variable            |
| Pe*         | Chemically Peculiar Star     |
| Ro*         | Rotating Variable            |
| RS*         | RS CVn Variable              |
| dS*         | delta Sct Variable           |
| V*          | Variable Star                |
| EB*         | Eclipsing Binary             |
| gD*         | gamma Dor Variable           |
| TT*         | T Tauri Star                 |
| Y*O         | Young Stellar Object         |
| EB?         | Eclipsing Binary             |
| El*         | Ellipsoidal Variable         |
| Pu*         | Pulsating Variable           |
| LM*         | Low-mass Star                |
| HV*         |  High Velocity Star          |
| Ir*         | Irregular Variable           |
| RR?         | RR Lyrae Variable            |
| SB?         | Spectroscopic Binary         |


Only bona-fide single stars (`PM*` or `*`) are kept, remaining emission-line objects, eclipsing binaries, etc. are discarded.

In [6]:
# Export star names for external SIMBAD query (run once)
df["Name"] = df["Name"].str.strip()
with open(STAR_NAMES, "w") as fh:
    for name in df["Name"]:
        fh.write(name + "\n")

# Load pre-computed SIMBAD results
simbad = pd.read_csv(SIMBAD_FILE, delimiter="|")
simbad.columns = simbad.columns.str.strip()
simbad = simbad.rename(
    columns={
        "typed ident": "Name",
        "typ": "obj_type",
        "spec. type": "spec_type",
    }
)
for col in ["Name", "obj_type", "spec_type"]:
    simbad[col] = simbad[col].str.strip()

df = df.merge(simbad[["Name", "obj_type", "spec_type"]], on="Name", how="left")

print("SIMBAD object-type distribution:")
print(df["obj_type"].value_counts().to_string())

SIMBAD object-type distribution:
obj_type
PM*    4562
*      1788
SB*     149
Em*      74
**       57
BY*      49
Er*      21
Pe*      17
RS*       7
dS*       7
EB*       6
V*        6
Ro*       5
gD*       4
TT*       3
EB?       2
Y*O       2
El*       1
Pu*       1
LM*       1
HV*       1
Ir*       1
RR?       1
SB?       1


In [7]:
# Keep only confirmed single stars
VALID_TYPES = ["PM*", "*"]
df = df[df["obj_type"].isin(VALID_TYPES)].copy()
print(f"After SIMBAD filtering: {len(df):,} stars")

After SIMBAD filtering: 6,350 stars


## 6. Final dataset

In [8]:
df_final = df[SELECTED_COLUMNS].reset_index(drop=True).rename(columns=COLUMN_RENAME)

# Keep only stars whose SIMBAD spectral type starts with F or G
df_final = df_final[
    df_final["SpecType"].str.startswith(("F", "G"), na=False)
].reset_index(drop=True)

df_F = df_final[df_final["SpecType"].str.startswith("F")].reset_index(drop=True)
df_G = df_final[df_final["SpecType"].str.startswith("G")].reset_index(drop=True)

print(f"All F+G stars: {len(df_final):,}")
print(f"  F-type: {len(df_F):,}")
print(f"  G-type: {len(df_G):,}")

All F+G stars: 6,014
  F-type: 3,450
  G-type: 2,564


In [9]:
df_final.to_csv(OUT_ALL, index=False)
df_F.to_csv(OUT_F, index=False)
df_G.to_csv(OUT_G, index=False)

print(f"Saved: {OUT_ALL}")
print(f"Saved: {OUT_F}")
print(f"Saved: {OUT_G}")

Saved: ../data/processed/gcs-allstars.csv
Saved: ../data/processed/gcs-Fstars.csv
Saved: ../data/processed/gcs-Gstars.csv


In [10]:
df_final.head(3)

,Name,fs,RAh,RAm,RAs,DE_,DEd,DEm,DEs,GLON,...,Rmax,ecc,zmax,RA,DEC,X,Y,Z,ObjType,SpecType
0,HD 23,,0,5,7.4,-,52,9,6,319,...,8.88,0.16,0.15,1.280833,-52.151667,13.895403,-12.079091,-37.749352,PM*,G0V
1,HD 59,,0,5,33.5,+,46,39,46,115,...,8.31,0.24,0.33,1.389583,46.662778,-35.514961,76.162064,-22.517258,PM*,G5
2,HD 67,*,0,5,28.4,-,61,13,33,313,...,7.98,0.09,0.14,1.368333,-61.225833,21.123623,-22.652313,-44.234211,PM*,G5V
